# FDA Controlled-Substance Support Audit**Project:** ECON 580 Thesis (Regulatory speed and diversion risk)  **Date:** 2026-03-08This notebook performs a **feasibility audit**: do FDA approvals have a sufficiently large and meaningful overlap with a plausible controlled-substance universe to justify an FDA × downstream-outcome design?

## Purpose of the Audit- Estimate the overlap between FDA approvals and likely controlled substances.- Assess whether the overlap is large, time-spanning, and heterogeneous enough for a downstream-outcome design.- Document classification uncertainty and data limitations.

## Why This Matters for the ThesisIf the FDA approval sample barely overlaps the universe of controlled substances, downstream diversion outcomes (e.g., ARCOS, NFLIS) may not align with the FDA approval-speed sample. This audit is designed to avoid investing in a mismatch.

## Data Used- `data/processed/fda_backbone.csv`: FDA ORIG submission backbone from Drugs@FDA data files.- Output tables and figures are saved to `output/tables/` and `output/figures/`.**Important limitation:** This audit uses **heuristic name matching** only. No DEA schedule file is linked in this notebook.

## Screening Logic (Conservative)We classify each FDA approval into one of four categories using **token-based name matching** on active ingredients:- `likely_controlled_substance`: matches a conservative set of high-confidence controlled substances (opioids, stimulants, benzodiazepines, barbiturates, etc.).- `possible_controlled_substance`: matches a smaller set of lower-confidence or less well-known scheduled substances.- `unclear`: ambiguous cases or missing active ingredient information.- `unlikely_controlled_substance`: no controlled-substance tokens matched.This approach **prefers false negatives** (missing true controlled substances) to false positives.

In [ ]:
from pathlib import Pathimport sysimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltROOT = Path("..").resolve()sys.path.append(str(ROOT / "src"))from support_audit import (    add_screening_columns,    create_summary_tables,    save_summary_tables,    make_figures,    LIKELY_CONTROLLED_TOKENS,    POSSIBLE_CONTROLLED_TOKENS,    UNCLEAR_TOKENS,)pd.set_option("display.max_columns", 50)

### Load FDA Backbone

In [ ]:
backbone_path = ROOT / "data" / "processed" / "fda_backbone.csv"backbone = pd.read_csv(backbone_path)backbone.shape

### Create Screening Dataset and Apply Classification

In [ ]:
screening_cols = [    "appl_no",    "appl_type",    "submission_no",    "approval_date",    "approval_year",    "review_priority_group",    "priority_review",    "orphan_designation",    "sponsor_name",    "drug_name",    "active_ingredient",    "active_ingredient_norm",]screening_df = backbone[screening_cols].copy()screening_df = add_screening_columns(screening_df)screening_df.head()

### Save Screened Candidates

In [ ]:
output_path = ROOT / "data" / "intermediate" / "controlled_substance_screened_candidates.csv"screening_df.to_csv(output_path, index=False)output_path

## Validation and Spot Checks

In [ ]:
# Row count checkprint("Backbone rows:", backbone.shape[0])print("Screened rows:", screening_df.shape[0])# Duplicate check on appl_no + submission_nodup_count = screening_df.duplicated(subset=["appl_no", "submission_no"]).sum()print("Duplicate appl_no + submission_no rows:", dup_count)# Screening distributionscreening_df["controlled_substance_screen"].value_counts()

In [ ]:
# Spot-check examples from each classificationsamples = (    screening_df    .sort_values("approval_year")    .groupby("controlled_substance_screen")    .head(5)    [["appl_no", "approval_year", "drug_name", "active_ingredient", "controlled_substance_screen", "notes"]])samples

## Summary Tables and Figures

In [ ]:
from pathlib import Pathtables = create_summary_tables(screening_df)output_tables_dir = ROOT / "output" / "tables"output_figures_dir = ROOT / "output" / "figures"save_summary_tables(tables, output_tables_dir)make_figures(screening_df, tables, output_figures_dir)list(tables.keys())

In [ ]:
# Display key tablesprint(tables["controlled_substance_support_summary"].head())print(tables["controlled_substance_by_year"].head())print(tables["controlled_substance_priority_summary"].head())

## Support Audit: Is There Enough Overlap for an FDA × Downstream-Outcome Design?

In [ ]:
from IPython.display import Markdownsubset_lp = screening_df[    screening_df["controlled_substance_screen"].isin([        "likely_controlled_substance",        "possible_controlled_substance",    ])].copy()count_total = screening_df.shape[0]count_lp = subset_lp.shape[0]share_lp = count_lp / count_total if count_total else 0by_year = tables["controlled_substance_by_year"].copy()nonzero = by_year[by_year["total_likely_possible"] > 0]min_year = int(nonzero["approval_year"].min()) if not nonzero.empty else Nonemax_year = int(nonzero["approval_year"].max()) if not nonzero.empty else Nonepriority_counts = subset_lp["review_priority_group"].value_counts(dropna=False)orphan_counts = subset_lp["orphan_designation"].value_counts(dropna=False)ingredients = subset_lp["active_ingredient"].dropna().str.split(";").explode().str.strip()counts = ingredients.value_counts()unique_ingredients = ingredients.nunique()top10_share = counts.head(10).sum() / counts.sum() if counts.sum() else 0memo = f'''### Summary- **Likely + possible controlled-substance overlap:** {count_lp} approvals (~{share_lp:.1%} of FDA ORIG approvals).- **Time span:** {min_year}–{max_year} with nonzero overlap (coverage across {len(nonzero)} distinct years).- **Priority variation:** PRIORITY={priority_counts.get('PRIORITY', 0)}, STANDARD={priority_counts.get('STANDARD', 0)}, MISSING={priority_counts.get('MISSING', 0)}. Missing review-priority data remain substantial.- **Orphan designation:** {orphan_counts.get(True, 0)} of {count_lp} (~{(orphan_counts.get(True, 0)/count_lp if count_lp else 0):.1%}). Orphan does **not** dominate the overlap set.- **Ingredient concentration:** {unique_ingredients} unique ingredients; top 10 account for ~{top10_share:.1%} of ingredient mentions (mix includes combination products).### Interpretation- The overlap is **non-trivial but modest** (~7–8% of ORIG approvals), with long time coverage.- Review priority variation exists but is **partly masked by missing priority data**.- The overlap is **not** dominated by orphan drugs.- Ingredient counts show **concentration in a handful of substances**, which suggests a narrower empirical universe.### Best-fit downstream outcome- Most promising for **(a) ARCOS-style legal distribution outcomes**, especially if focused on opioids/stimulants/benzodiazepines.- NFLIS-style illicit-market indicators may be feasible but would likely require a narrower drug-class focus.- A **narrower drug-class thesis** (option d) appears more defensible than a broad, all-drug design.### Recommended next-step sample definition- Start with the **likely-controlled subset only**, and restrict to clear opioid/stimulant/benzodiazepine tokens.- Focus on approval years with more complete review-priority reporting (post-1992 or later).- Add DEA schedule data (or ARCOS reference files) to convert this heuristic screen into a verified controlled-substance sample.'''Markdown(memo)